# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


## Model Choice

For this project, the baseline model is a **Linear Surrogate operating on an SVD-reduced latent space**. 

Before developing complex Physics-Informed Neural Networks (PINNs) to capture highly non-linear dynamics, it is essential to establish a classical, purely data-driven linear baseline. In traditional finite element method (FEM) simulations (such as those run in Abaqus or COMSOL), high-fidelity models contain massive degrees of freedom. A standard Model Order Reduction (MOR) technique uses Singular Value Decomposition (SVD)—also known as Proper Orthogonal Decomposition (POD) in mechanics—to project this high-dimensional space onto a low-dimensional subspace. 

By coupling SVD with a simple Linear Regression model, we test the hypothesis: *Can the temporal evolution of these dominant physical modes be accurately predicted using a linear mapping?* If the material behavior or boundary conditions introduce significant non-linearities, this baseline will underfit, rigorously justifying the transition to advanced neural network architectures.

## Feature Selection

Instead of training a model on the raw spatial coordinates of 10,000 mesh nodes, we extract the governing physical features using **Singular Value Decomposition (SVD)**.

* **Input Features ($X$):** Discrete time steps ($t$).
* **Target Features ($y$):** The truncated latent space variables (the top $r=2$ dominant SVD modes). In materials science, these modes represent the principal kinematic or thermal deformations of the system. By truncating the matrices, we filter out high-frequency spatial noise and retain the core thermodynamic or mechanical variance.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ---------------------------------------------------------
# 1. Synthetic FEM Data Generation & Feature Selection (SVD)
# ---------------------------------------------------------
n_nodes = 10000  
n_steps = 50     

x_space = np.linspace(0, 10, n_nodes)
t_time = np.linspace(0, 5, n_steps)
X, T = np.meshgrid(x_space, t_time)

FEM_data = (np.sin(X) * np.exp(-0.2 * T)).T 

U, S, Vt = np.linalg.svd(FEM_data, full_matrices=False)

r = 2  
U_r = U[:, :r]
S_r = np.diag(S[:r])
Vt_r = Vt[:r, :]

latent_trajectories = (S_r @ Vt_r).T 
time_inputs = t_time.reshape(-1, 1)


### Code Breakdown & Materials Science Connection

**Libraries Used:**
* `numpy`: The foundational library for high-performance matrix operations, critical for handling large tensors analogous to FEM stiffness/mass matrices.
* `matplotlib.pyplot`: Used for 2D plotting to visualize physical field reconstructions.
* `sklearn.linear_model.LinearRegression`: Implements Ordinary Least Squares regression to map our time vector to our latent state.
* `sklearn.metrics.mean_squared_error`: Computes the $L_2$ norm-based loss to quantify surrogate accuracy.

**Line-by-Line Logic:**
* `n_nodes = 10000` & `n_steps = 50`: Simulates a discretized physical domain (e.g., a 1D rod or thermal gradient) with 10,000 finite elements evaluated over 50 time increments.
* `x_space = np.linspace(...)` & `t_time = ...`: Defines the physical boundary and temporal simulation window.
* `X, T = np.meshgrid(x_space, t_time)`: Creates a Cartesian grid out of the spatial and temporal vectors, allowing vectorized calculation of the physical state.
* `FEM_data = (np.sin(X) * np.exp(-0.2 * T)).T`: Generates the synthetic ground truth. Physically, this equation mimics a transient diffusion process (like heat dissipating) or a damped standing wave. The `.T` transposes the matrix so rows represent spatial nodes and columns represent time steps (a standard matrix shape for MOR).
* `U, S, Vt = np.linalg.svd(...)`: Performs Singular Value Decomposition. 
    * `U` contains the spatial modes (orthogonal spatial basis functions).
    * `S` contains the singular values (energy/variance captured by each mode).
    * `Vt` contains the temporal evolution of these modes.
* `r = 2`: We truncate the physics to only the top 2 energy-dominant modes, vastly reducing the computational burden.
* `U_r = U[:, :r]` ... `Vt_r = Vt[:r, :]`: Slices the matrices to extract the reduced basis.
* `latent_trajectories = (S_r @ Vt_r).T`: Computes the actual target values our machine learning model needs to predict. We multiply the singular values by the temporal matrix to get the time-varying coefficients for our spatial modes.
* `time_inputs = t_time.reshape(-1, 1)`: Reshapes our 1D time array into a 2D column vector, which is strictly required by Scikit-Learn's API.


## Implementation

Here, we initialize the linear surrogate and map the temporal domain to the reduced latent space.

In [ ]:
# ---------------------------------------------------------
# 2. Initialize and train the baseline model
# ---------------------------------------------------------
baseline_model = LinearRegression()
baseline_model.fit(time_inputs, latent_trajectories)

predicted_latent = baseline_model.predict(time_inputs)

reconstructed_field = U_r @ predicted_latent.T


### Code Breakdown & Materials Science Connection

**Line-by-Line Logic:**
* `baseline_model = LinearRegression()`: Initializes the linear model.
* `baseline_model.fit(time_inputs, latent_trajectories)`: Trains the model. It finds the optimal linear weights to map a specific time $t$ to the coefficients of our $r=2$ dominant spatial modes.
* `predicted_latent = baseline_model.predict(time_inputs)`: We pass our time vector back into the trained model to generate the predicted coefficients.
* `reconstructed_field = U_r @ predicted_latent.T`: The crucial reconstruction step. By taking the dot product (`@`) of our high-dimensional spatial basis (`U_r`) and the model's predicted temporal coefficients, we project the low-dimensional prediction back into the full 10,000-node physical space. This outputs a predicted field state that can be directly compared against the original heavy FEM solver output.


## Evaluation

To validate the surrogate against the full-order model, we utilize the **Mean Squared Error (MSE)** and the **Relative $L_2$ Error**. In computational mechanics, the relative $L_2$ error is the gold standard for comparing a reduced-order model against an exact FEM solution, as it normalizes the error against the total energy of the true physical field. We also plot the spatial reconstruction at a specific temporal snapshot to qualitatively assess the fit.

In [ ]:
# ---------------------------------------------------------
# 3. Evaluate the baseline model
# ---------------------------------------------------------
mse = mean_squared_error(FEM_data, reconstructed_field)
relative_l2 = np.linalg.norm(FEM_data - reconstructed_field) / np.linalg.norm(FEM_data)

print(f"Mean Squared Error (MSE): {mse:.6f}")
print(f"Relative L2 Error: {relative_l2:.6f}")

time_idx = 25
plt.figure(figsize=(10, 5))
plt.plot(x_space, FEM_data[:, time_idx], label="Ground Truth (FEM)", color='black', linestyle='--')
plt.plot(x_space, reconstructed_field[:, time_idx], label="Baseline (SVD + Linear Reg)", color='red', alpha=0.7)
plt.title(f"Physical Field Reconstruction at t = {t_time[time_idx]:.2f}s")
plt.xlabel("Spatial Domain (Nodes)")
plt.ylabel("Physical State (e.g., Temperature / Displacement)")
plt.legend()
plt.grid(True)
plt.show()


### Code Breakdown & Materials Science Connection

**Line-by-Line Logic:**
* `mse = mean_squared_error(...)`: Computes the average squared difference between the true FEM matrix and our reconstructed matrix.
* `relative_l2 = np.linalg.norm(...) / np.linalg.norm(...)`: Calculates the Frobenius norm of the residual (the error matrix) divided by the norm of the true matrix. This provides a percentage-like metric of global error across the entire spatio-temporal domain.
* `time_idx = 25`: Selects the 25th time step (the midpoint of our transient simulation) for visual comparison.
* `plt.plot(x_space, FEM_data[:, time_idx], ...)`: Plots the exact solution across all 10,000 spatial nodes at $t=2.5s$.
* `plt.plot(x_space, reconstructed_field[:, time_idx], ...)`: Overlays our baseline model's prediction. If the transient physics are strictly linear, the red curve will perfectly mask the black dashed line. If the underlying physics contain non-linearities (such as plasticity in solid mechanics or turbulent convection), this linear baseline will fail to match the curve, thus proving the necessity for the non-linear PyTorch PINN implemented in the subsequent stages of this project.
